# Collect Shared DOCX Files into a Drive Folder

Find all Word (.docx) files that a specific person shared with you in Google Drive and put them (or shortcuts to them) in one folder, so they are easy to find. It does a dry run first so you can check the list before anything changes.

## How to run
1. Run the cells and sign in when Colab asks.
2. Enter the owner's email and the folder name in the form cell.
3. Check the dry-run list, then set `CONFIRM_MOVE = True` and run the last cell.

_Rewritten from my original Colab notebook: `docx filter and move.ipynb`._

In [ ]:
!pip install -q --upgrade google-api-python-client

In [ ]:
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

auth.authenticate_user()
drive_service = build("drive", "v3")
print("Drive API ready")

In [ ]:
OWNER_EMAIL = ""  #@param {type:"string"}
TARGET_FOLDER_NAME = "Shared DOCX"  #@param {type:"string"}

## Create (or reuse) the target folder in My Drive

In [ ]:
def get_or_create_folder(name):
    query = (f"name='{name}' and mimeType='application/vnd.google-apps.folder' "
             "and 'root' in parents and trashed=false")
    found = drive_service.files().list(q=query, fields="files(id, name)").execute().get("files", [])
    if found:
        return found[0]["id"]
    meta = {"name": name, "mimeType": "application/vnd.google-apps.folder"}
    return drive_service.files().create(body=meta, fields="id").execute()["id"]

folder_id = get_or_create_folder(TARGET_FOLDER_NAME)
print("Folder ID:", folder_id)

## Dry run: list matching files

In [ ]:
DOCX_MIME = "application/vnd.openxmlformats-officedocument.wordprocessingml.document"

def find_shared_docx(owner_email):
    query = (f"mimeType='{DOCX_MIME}' and '{owner_email}' in owners "
             "and sharedWithMe=true and trashed=false")
    results, page_token = [], None
    while True:
        resp = drive_service.files().list(
            q=query, fields="nextPageToken, files(id, name)", pageToken=page_token
        ).execute()
        results.extend(resp.get("files", []))
        page_token = resp.get("nextPageToken")
        if not page_token:
            return results

matches = find_shared_docx(OWNER_EMAIL)
print(f"Found {len(matches)} files")
for f in matches:
    print(" -", f["name"])

## Add files to the folder
Files owned by someone else usually cannot be moved, so for those we create a shortcut in the folder instead.

In [ ]:
CONFIRM_MOVE = False  # change to True after checking the list above

if CONFIRM_MOVE:
    for f in matches:
        try:
            drive_service.files().update(fileId=f["id"], addParents=folder_id, fields="id").execute()
            print("Added:", f["name"])
        except HttpError:
            shortcut = {"name": f["name"], "mimeType": "application/vnd.google-apps.shortcut",
                        "shortcutDetails": {"targetId": f["id"]}, "parents": [folder_id]}
            drive_service.files().create(body=shortcut, fields="id").execute()
            print("Shortcut created:", f["name"])
else:
    print("Dry run only. Set CONFIRM_MOVE = True to continue.")